# 01 — Exploratory Data Analysis

IEEE-CIS Fraud Detection dataset.

What this notebook covers:
1. Dataset shape and column overview
2. Class imbalance — the 3.5% problem
3. Transaction amount distribution
4. Time patterns (hour of day, day of week)
5. Null rate heatmap — understanding missingness
6. Top fraud signals (quick correlation scan)

**LinkedIn post angle:** *'3 things about the IEEE-CIS fraud dataset that will break your model if you ignore them.'*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = Path('../data')

## 1. Load data

In [ ]:
# Load from Parquet (faster than CSV, run make ingest first)
try:
    trans = pd.read_parquet(DATA_DIR / 'processed/transactions.parquet')
    identity = pd.read_parquet(DATA_DIR / 'processed/identity.parquet')
    print(f'Transactions: {trans.shape}')
    print(f'Identity:     {identity.shape}')
except FileNotFoundError:
    print('Run `make ingest` first to generate Parquet files.')

## 2. Class imbalance — the core challenge

In [ ]:
fraud_counts = trans['isFraud'].value_counts()
print(f'Legitimate: {fraud_counts[0]:,} ({fraud_counts[0]/len(trans):.1%})')
print(f'Fraud:      {fraud_counts[1]:,} ({fraud_counts[1]/len(trans):.1%})')

fig = px.pie(
    values=fraud_counts.values,
    names=['Legitimate', 'Fraud'],
    title='Class Distribution — 3.5% fraud rate',
    color_discrete_sequence=['#636EFA', '#EF553B']
)
fig.show()

# Key insight: a naive 'predict everything as legit' model gets 96.5% accuracy.
# This is why we use PR-AUC, not accuracy or ROC-AUC.

## 3. Transaction amount distribution

In [ ]:
fig = px.histogram(
    trans,
    x='TransactionAmt',
    color='isFraud',
    nbins=100,
    log_y=True,
    title='Transaction Amount Distribution (log scale)',
    labels={'isFraud': 'Is Fraud', 'TransactionAmt': 'Amount (USD)'},
    color_discrete_map={0: '#636EFA', 1: '#EF553B'}
)
fig.show()

print('Fraud amount stats:')
print(trans[trans['isFraud']==1]['TransactionAmt'].describe())
print('\nLegitimate amount stats:')
print(trans[trans['isFraud']==0]['TransactionAmt'].describe())

## 4. Time patterns

In [ ]:
REFERENCE_DATE = pd.Timestamp('2017-11-30')
trans['datetime'] = REFERENCE_DATE + pd.to_timedelta(trans['TransactionDT'], unit='s')
trans['hour'] = trans['datetime'].dt.hour
trans['dayofweek'] = trans['datetime'].dt.dayofweek

# Fraud rate by hour of day
hourly_fraud = trans.groupby('hour')['isFraud'].mean().reset_index()
fig = px.bar(
    hourly_fraud,
    x='hour',
    y='isFraud',
    title='Fraud Rate by Hour of Day',
    labels={'isFraud': 'Fraud Rate', 'hour': 'Hour (UTC)'}
)
fig.show()

# Key insight: night hours (0-5am) have higher fraud rates.
# This is why we add is_night as a feature.

## 5. Null rate heatmap

In [ ]:
null_rates = trans.isnull().mean().sort_values(ascending=False)
print(f'Columns with >50% nulls: {(null_rates > 0.5).sum()}')
print(f'Columns with >90% nulls: {(null_rates > 0.9).sum()}')
print()
print('Top 20 null columns:')
print(null_rates.head(20).to_string())

# Key insight: V-columns dominate the high-null list.
# Many are >75% null. We can't drop them (they're informative) but we can't
# impute blindly either. Our strategy: null indicator + median imputation.

## 6. Quick correlation scan — what predicts fraud?

In [ ]:
# Point-biserial correlation with isFraud for numeric columns
numeric_cols = trans.select_dtypes(include=[np.number]).columns.tolist()
corr_with_fraud = trans[numeric_cols].corrwith(trans['isFraud']).abs().sort_values(ascending=False)

print('Top 20 features correlated with fraud:')
print(corr_with_fraud.drop('isFraud').head(20).to_string())